# 11 — Does the group proxy actually recover patients?

**This is the notebook that decides how strong a claim the paper may make about IQ-OTH/NCCD.**

IQ-OTH/NCCD has no subject IDs, so the recovered grouping cannot be checked against ground truth
there. The fix: take a *different* CT dataset that **does** have subject IDs, throw them away, run
the identical changepoint recovery, and score the result against the truth we hid.

| Metric | Meaning | What a good result looks like |
|---|---|---|
| Adjusted Rand Index | agreement with true subject partition | > 0.6 strong, > 0.3 usable, ~0 failed |
| purity | fraction of slices whose block majority is their own subject | high |
| homogeneity | each recovered block holds one subject | high |
| completeness | each subject lands in one block | high |
| **true subjects straddling** | **the metric that actually matters** | **0 under `block_shuffled`** |

Note the last row. We do not need recovered blocks to *equal* subjects. We need them to stop
subjects from straddling partitions. A proxy can score a mediocre ARI and still be perfectly
adequate for split hygiene, and this notebook measures that directly.

**A low ARI here is a publishable negative result** — it would mean the IQ-OTH grouping is
unvalidated and the paper must say so. Do not tune until it passes.

CPU-mostly, ~20 min.

In [ ]:
# ── environment ──────────────────────────────────────────────────────────
# 1. Add the dataset:  hamdallak/the-iqothnccd-lung-cancer-dataset
# 2. Add your uploaded library dataset: iqoth-honest-lib  (upload src/iqoth/)
import sys, os, json, time, types

print("Mounted in /kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "no /kaggle/input")

LIB = None
ROOT = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "splits.py" in files and "audit.py" in files and not LIB:
            LIB = root
        if any("bengin" in d.lower() or "benign" in d.lower() for d in dirs) and not ROOT:
            ROOT = root

if LIB:
    pkg = types.ModuleType("iqoth")
    pkg.__path__ = [LIB]
    sys.modules["iqoth"] = pkg
    sys.path.insert(0, LIB)
else:
    sys.path.insert(0, "src")          # local fallback

if not ROOT:
    ROOT = os.environ.get("IQOTH_ROOT")

assert os.path.isdir(ROOT or ""), f"dataset not found; /kaggle/input tree: {list(os.walk('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'no input'}"

RESULTS = "/kaggle/working/results"
os.makedirs(RESULTS, exist_ok=True)

from iqoth import splits, audit, metrics
print("lib:", LIB)
print("dataset:", ROOT)

## Choose an external dataset

Needs: CT (or at least medical) images, **genuine subject/patient IDs**, and small enough for
Kaggle. Verify availability yourself — slugs change. Candidates, in order of preference:

1. **SARS-CoV-2 CT-Scan** (Soares et al.) — ~2,482 slices, 120 patients, patient metadata CSV.
2. **COVID-CT** (UCSD, Zhao et al.) — ~750 slices with patient IDs in the metadata.
3. **LUNA16 / LIDC-IDRI 2D exports** — real `seriesuid` per scan; use a preprocessed Kaggle mirror,
   not the 125 GB original.
4. **MosMedData** — per-study NIfTI volumes; each study is one subject.

Pick one, set `EXT_ROOT`, and configure the matching adapter below.

In [ ]:
from iqoth import external
import csv, glob
from PIL import Image
import numpy as np

# 1. Locate LUNA16 in /kaggle/input
LUNA_ROOT = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "annotations.csv" in files and not LUNA_ROOT:
            LUNA_ROOT = root
assert LUNA_ROOT, f"LUNA16 dataset not found in /kaggle/input; tree: {list(os.walk('/kaggle/input'))}"
print("Found LUNA16 root:", LUNA_ROOT)

# 2. Load annotations.csv to classify series into nodule vs normal
nodule_uids = set()
ann_file = os.path.join(LUNA_ROOT, "annotations.csv")
if os.path.exists(ann_file):
    with open(ann_file) as f:
        for row in csv.DictReader(f):
            nodule_uids.add(row["seriesuid"])
print(f"Total nodule-positive UIDs in annotations.csv: {len(nodule_uids)}")

# 3. Find .mhd files in subset0 and subset1
mhd_files = []
for sdir in ("subset0", "subset1"):
    p = os.path.join(LUNA_ROOT, sdir)
    if os.path.isdir(p):
        mhd_files.extend(glob.glob(f"{p}/**/*.mhd", recursive=True))
if not mhd_files:
    mhd_files = glob.glob(f"{LUNA_ROOT}/**/*.mhd", recursive=True)
print(f"Found {len(mhd_files)} .mhd scans in LUNA16")

# Select cohort: balance nodule vs normal (up to 20 nodule + 20 normal)
nodule_scans = [f for f in mhd_files if os.path.splitext(os.path.basename(f))[0] in nodule_uids]
normal_scans = [f for f in mhd_files if os.path.splitext(os.path.basename(f))[0] not in nodule_uids]
cohort = nodule_scans[:15] + normal_scans[:15]
print(f"Selected cohort of {len(cohort)} scans ({len(nodule_scans[:15])} nodule, {len(normal_scans[:15])} normal)")

# 4. Slice .mhd/.raw into /kaggle/working/luna2d/<class>/<seriesuid>/
LUNA2D = "/kaggle/working/luna2d"
os.makedirs(LUNA2D, exist_ok=True)

def read_mhd_raw(mhd_path):
    header = {}
    with open(mhd_path, "r", errors="ignore") as f:
        for line in f:
            if "=" in line:
                k, v = line.split("=", 1)
                header[k.strip()] = v.strip()
    raw_name = header.get("ElementDataFile")
    raw_path = os.path.join(os.path.dirname(mhd_path), raw_name)
    dims = [int(x) for x in header.get("DimSize", "512 512 0").split()]
    depth, height, width = dims[2], dims[1], dims[0]
    arr = np.fromfile(raw_path, dtype=np.int16).reshape((depth, height, width))
    return arr

t0 = time.time()
for mhd in cohort:
    suid = os.path.splitext(os.path.basename(mhd))[0]
    cls = "nodule_cases" if suid in nodule_uids else "normal_cases"
    out_dir = os.path.join(LUNA2D, cls, suid)
    if os.path.isdir(out_dir) and len(os.listdir(out_dir)) > 20:
        continue
    os.makedirs(out_dir, exist_ok=True)
    vol = read_mhd_raw(mhd)
    vol = np.clip(vol, -1000, 400)
    vol = ((vol + 1000) / 1400.0 * 255.0).astype(np.uint8)
    for idx, sl in enumerate(vol):
        im = Image.fromarray(sl).resize((128, 128))
        im.save(os.path.join(out_dir, f"slice_{idx:04d}.png"))
print(f"Slicing complete in {(time.time()-t0):.1f}s")

# 5. Build records via folder_per_subject
ext = external.adapter_folder_per_subject(LUNA2D)
subs = {r["subject_id"] for r in ext}
print(f"images {len(ext)}   subjects {len(subs)}   classes {sorted({r['class_name'] for r in ext})}")
assert len(subs) > 5, "adapter found too few subjects"
assert len(ext) > len(subs), "need multiple slices per subject"
print(f"mean slices/subject {len(ext)/len(subs):.1f}")


## Recovery quality, per class

In [ ]:
report = {}
for cname in sorted({r["class_name"] for r in ext}):
    sub = [r for r in ext if r["class_name"] == cname]
    if len({r["subject_id"] for r in sub}) < 3:
        print(f"skip {cname}: too few subjects"); continue
    v = external.proxy_validation(sub)
    report[cname] = v
    print(f"\n### {cname}  ({v['n_images']} images, {v['true_n_subjects']} subjects, "
          f"{v['true_mean_slices_per_subject']} slices/subject)")
    for tag in ("known_k", "estimated_k"):
        m = v[tag]
        print(f"   {tag:<12} ARI={m['adjusted_rand_index']:6.3f}  purity={m['purity']:6.3f}  "
              f"hom={m['homogeneity']:5.3f}  comp={m['completeness']:5.3f}  "
              f"blocks={m['n_recovered_groups']}")
    print(f"   null control  ARI={v['null_control']['adjusted_rand_index']:6.3f}  "
          f"(must be ~0, else the pipeline is broken)")

## Changepoint recovery, and the metric that actually matters

`known_k` above uses equal cuts. Now run the **changepoint** recovery — the method the paper
proposes — and measure how many *true* subjects straddle partitions when the split is built on
recovered blocks.

In [ ]:
import numpy as np
from collections import defaultdict

cp_report = {}
for cname in sorted({r["class_name"] for r in ext}):
    sub = [r for r in ext if r["class_name"] == cname]
    k = len({r["subject_id"] for r in sub})
    if k < 3: continue
    info = external.refine_blocks_by_changepoint(sub, k, label_prefix=str(sub[0]["label"]))
    truth = [r["subject_id"] for r in sorted(sub, key=lambda r: r["pos"])]
    pred = [r["block_id"] for r in sorted(sub, key=lambda r: r["pos"])]
    ari = external.adjusted_rand_index(truth, pred)
    hom, com = external.homogeneity_completeness(truth, pred)
    cp_report[cname] = {"ari": round(ari, 4), "purity": round(external.purity(truth, pred), 4),
                        "homogeneity": round(hom, 4), "completeness": round(com, 4),
                        "prominence_ratio": round(info["prominence_ratio"], 3),
                        "block_sizes": info["block_sizes"]}
    print(f"{cname:<20} changepoint ARI={ari:6.3f}  prominence={info['prominence_ratio']:5.2f}x  "
          f"blocks={info['block_sizes']['n_blocks']}")

In [ ]:
leak = external.leakage_under_proxy(ext)
print("True subjects straddling partitions, split built on RECOVERED blocks:\n")
for strat, v in leak.items():
    print(f"  {strat:<18} {v['true_subjects_straddling']:>4} / {v['true_subjects']} "
          f"({v['pct_straddling']}%)")
print("\nVERDICT")
bs = leak.get("block_shuffled", {})
if bs.get("true_subjects_straddling", 99) == 0:
    print("  Recovered-block splits keep TRUE subjects intact -> the proxy is adequate for")
    print("  split hygiene even if ARI is imperfect. This is the claim the paper should make.")
else:
    print(f"  {bs.get('true_subjects_straddling')} true subjects still straddle. The proxy is NOT")
    print("  sufficient. Report this honestly and weaken the IQ-OTH claims accordingly.")

In [ ]:
with open(f"{RESULTS}/11_proxy_validation.json", "w") as f:
    json.dump({"equal_cut": report, "changepoint": cp_report, "leakage_under_proxy": leak}, f,
              indent=2, default=str)
print("saved")